# 02 — OCR chữ Hán bằng PaddleOCR (free, local, không giới hạn)

Chạy toàn bộ ảnh **miễn phí, không quota**. Dùng module `src/paddle_ocr.py`.

**Lưu ý kỹ thuật (đã kiểm chứng trên máy):**
- PaddleOCR 3.x + paddlepaddle 3.x trên CPU cần `enable_mkldnn=False` (tránh lỗi oneDNN). Đã set sẵn trong module.
- `lang='chinese_cht'` (phồn thể) gần với Hán cổ mộc bản nhất.
- Chữ viết dọc: sắp xếp cột **phải→trái** dựa trên toạ độ (đã xử lý trong `ocr_image`).

Vào: `data/raw_images/downloaded_images/<docid>/` (hoặc `preprocessed/`) →
Ra: `data/ocr/<docid>/*.txt` (từng trang) + `outputs/<docid>/<docid>_raw.txt` (gộp).

In [ ]:
# %pip install paddlepaddle>=3.0 paddleocr>=3.0

In [ ]:
import sys
from pathlib import Path
NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
sys.path.insert(0, str(PROJECT_ROOT / 'src'))   # để import module trong src/

DATA_DIR    = PROJECT_ROOT / 'data'
RAW_IMG_DIR = DATA_DIR / 'raw_images' / 'downloaded_images'   # ảnh gốc theo <docid>/
PREP_DIR    = DATA_DIR / 'preprocessed'
OCR_DIR     = DATA_DIR / 'ocr'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
for d in [PREP_DIR, OCR_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def list_works(base):
    """Danh sách 'work id' = tên thư mục con trong base (mỗi tác phẩm/quyển = 1 folder)."""
    if not base.exists():
        return []
    return sorted(p.name for p in base.iterdir() if p.is_dir())

def images_of(base, wid):
    d = base / wid
    return sorted(list(d.glob('*.jpg')) + list(d.glob('*.png')))

print('Project root:', PROJECT_ROOT)
print('Ảnh gốc     :', RAW_IMG_DIR)

In [ ]:
from paddle_ocr import make_ocr, ocr_image, avg_score
from tqdm.auto import tqdm

IMG_SOURCE = RAW_IMG_DIR      # đổi thành PREP_DIR nếu muốn dùng ảnh đã tiền xử lý
SAVE_VIZ = False              # True = lưu ảnh bounding box CẠNH file text (cùng thư mục)
RESULT_DIR = DATA_DIR / 'ocr_results'   # text + ảnh bbox lưu chung: <docid>/

ocr = make_ocr(lang='chinese_cht')   # lần đầu sẽ tự tải model (~vài chục MB)
works = list_works(IMG_SOURCE)
print(f'{len(works)} tác phẩm/quyển sẽ OCR.')

### Chạy OCR (có resume — trang đã OCR sẽ bỏ qua)

In [ ]:
for wid in works:
    page_dir = RESULT_DIR / wid; page_dir.mkdir(parents=True, exist_ok=True)
    imgs = images_of(IMG_SOURCE, wid)
    for ip in tqdm(imgs, desc=wid):
        txt_path = page_dir / (ip.stem + '.txt')
        if txt_path.exists():
            continue
        try:
            vdir = page_dir if SAVE_VIZ else None   # ảnh bbox lưu cùng chỗ với text
            text, items = ocr_image(ocr, ip, viz_dir=vdir)
        except Exception as e:
            print('  ! lỗi', ip.name, str(e)[:80]); text = ''
        txt_path.write_text(text, encoding='utf-8')

    # gộp trang -> outputs/<wid>/<wid>_raw.txt
    out_dir = OUTPUTS_DIR / wid; out_dir.mkdir(parents=True, exist_ok=True)
    pages = sorted(page_dir.glob('*.txt'))
    merged = []
    for pp in pages:
        merged.append(f'### {pp.stem} ###')
        merged.append(pp.read_text(encoding='utf-8'))
    (out_dir / f'{wid}_raw.txt').write_text('\n'.join(merged), encoding='utf-8')
    print(f'{wid}: OCR {len(pages)} trang -> {out_dir}/{wid}_raw.txt')

### Xem thử 1 trang

In [ ]:
if works:
    sample = sorted((RESULT_DIR / works[0]).glob('*.txt'))
    if sample:
        print(sample[0].name, '\n' + '-'*40)
        print(sample[0].read_text(encoding='utf-8')[:800])